# Reflexive KG — Hackathon Demo Notebook

**VOR Hackathon 2025 · UseCase 3**

Prerequisites (run once before this notebook):

```bash
python 01_schema.py
python 02_seed_data.py
python 05_synthesize_lifecycle.py
```

Embeddings are pre-computed so this notebook finishes in under 5 minutes (AC 18).

In [ ]:
import time
import importlib
import matplotlib.pyplot as plt
from neo4j import GraphDatabase

from config import NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD
from agents.lifecycle_doer import LifecycleDoer, COHORT

evaluate = importlib.import_module("06_evaluate")
pipeline = importlib.import_module("04_agent_pipeline")

print("Neo4j:", NEO4J_URI)

## 1. Seeded-anomaly evaluation (AC 3, 4, 14)

In [ ]:
metrics, ok = evaluate.report()
assert ok

## 2. Scenario 1 — Brand mismatch cascade (mandatory, AC 6)

In [ ]:
result = pipeline.run_lifecycle_pipeline(1, use_llm=False)
best = result.critic_result.best()
print(f"Latency: {result.latency_seconds}s | Confidence: {best.confidence:.3f}")
print(best.reasoning)
for node in best.path[:6]:
    print(f"  [{node.label}] {node.display_name}  score={node.anomaly_score}")

## 3. Scenario 4 — A/B: closed world vs Reflexive KG (mandatory, AC 9)

In [ ]:
pipeline.run_scenario_4_ab(use_llm=False)

## 4. Scenario 3 — Top-20 proactive risk ranking (AC 8)

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session() as session:
    ranked = LifecycleDoer(session).risk_rank(20)
driver.close()

for i, (sku, score) in enumerate(ranked, 1):
    print(f"{i:>2}. {sku:>10}  anomaly={score:.3f}")

## 5. Anomaly score distribution

In [ ]:
m = evaluate.evaluate()
scores = [v for _, v in m["ranked"]]
plt.figure(figsize=(8, 4))
plt.hist(scores, bins=30, color="#4fffb0", edgecolor="#111318")
plt.axvline(m["threshold"], color="#ff6b6b", linestyle="--",
            label=f"top-decile ({m['threshold']:.2f})")
plt.title(f"Anomaly scores — {COHORT}")
plt.xlabel("1 - cos(self_emb, reflect_emb)")
plt.ylabel("SKU count")
plt.legend()
plt.show()

## 6. Scenarios 5 & 6 + latency benchmark (AC 15, 16, 20)

In [ ]:
t0 = time.time()
bench = []
for num in [1, 2, 3, 4, 5, 6]:
    r = pipeline.run_lifecycle_pipeline(num, use_llm=False)
    bench.append((num, r.latency_seconds, len(r.critic_result.validated)))
elapsed = time.time() - t0

print(f"{'Scenario':<10} {'Latency':>8} {'Accepted':>9}")
for num, lat, acc in bench:
    print(f"{num:<10} {lat:>7.2f}s {acc:>9}")
print(f"\nSegment elapsed: {elapsed:.1f}s")